# Hierarchical Prediction

This notebook applies the trained hierarchical classification models to buildings that remain unresolved after the rule-based stages. Binary and subtype classifiers are applied sequentially to produce the final Level-1 and Level-2 predictions.

**Input**
- Trained hierarchical classification models and selected probability thresholds.
- Preprocessed prediction datasets for unresolved buildings and buildings requiring subtype classification.

**Output**
- Final Level-1 and Level-2 classifications for the machine-learning stage.
- Classification confidence and source information for the predicted buildings.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib
import os
import warnings

import time
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Directory config ──────────────────────────────────────────────────────────
ROOT_DIR  = '/fast/home/o-olajuyigbe/osm_project'
DATA_DIR  = os.path.join(ROOT_DIR, 'data')
ML_DIR    = os.path.join(DATA_DIR, 'ml_ready')
MODEL_DIR = os.path.join(DATA_DIR, 'models')
OUT_DIR   = os.path.join(DATA_DIR, 'final')
PROC_DIR  = os.path.join(DATA_DIR, 'processed')

# building file needed to validate that all building IDs are present in the final output
BUILDING_FILE = os.path.join(PROC_DIR, 'germany_buildings_feature_engineered.parquet')

FINAL_OUTPUT_FILE = os.path.join(OUT_DIR, 'germany_buildings_classified_final.parquet')

print(f"ML artefacts  : {ML_DIR}")
print(f"Model dir     : {MODEL_DIR}")
print(f"Output file   : {FINAL_OUTPUT_FILE}")


ML artefacts  : /fast/home/o-olajuyigbe/osm_project/data/ml_ready
Model dir     : /fast/home/o-olajuyigbe/osm_project/data/models
Output file   : /fast/home/o-olajuyigbe/osm_project/data/final/germany_buildings_classified_final.parquet


In [15]:
# The cascade levels in the order they are applied
LEVELS_xgb = [
    'l1b_ind_binary',
    'l2b_comm_binary',     # commercial vs civic_other (within non-res, non-ind)
    'l1a_res_subtypes',
    
]

LEVELS_lgb = [
        'l0_res_binary', 
    'l2a_ind_subtypes', 
    'l3a_comm_subtypes',   # office / retail / food_drink / accommodation
]

LEVELS = [
    'l0_res_binary',       # residential vs non-residential
    'l1a_res_subtypes',    # SFH / MFH / apartment / …
    'l1b_ind_binary',      # industrial vs non-industrial (within non-res)
    'l2a_ind_subtypes',    # storage / manufacturing / energy / …
    'l2b_comm_binary',     # commercial vs civic_other (within non-res, non-ind)
    'l3a_comm_subtypes',   # office / retail / food_drink / accommodation
]

models = {}
for level in LEVELS_lgb:
    model_path = os.path.join(MODEL_DIR, f'{level}_lgb_PRODUCTION_FINAL.pkl')
    models[level] = joblib.load(model_path)
    print(f"  Loaded: {level:<25}  ({type(models[level]).__name__})")

for level in LEVELS_xgb:
    model_path = os.path.join(MODEL_DIR, f'{level}_xgb_PRODUCTION_FINAL.pkl')
    
    models[level] = joblib.load(model_path)
    print(f"  Loaded: {level:<25}  ({type(models[level]).__name__})")

print(f"\nAll {len(models)} models loaded successfully.")

  Loaded: l0_res_binary              (LGBMClassifier)
  Loaded: l2a_ind_subtypes           (LGBMClassifier)
  Loaded: l3a_comm_subtypes          (LGBMClassifier)
  Loaded: l1b_ind_binary             (XGBClassifier)
  Loaded: l2b_comm_binary            (XGBClassifier)
  Loaded: l1a_res_subtypes           (XGBClassifier)

All 6 models loaded successfully.


In [16]:
le_l1   = joblib.load(os.path.join(ML_DIR, 'label_encoder_l1.pkl'))
le_l2   = joblib.load(os.path.join(ML_DIR, 'label_encoder_l2.pkl'))
le_res  = joblib.load(os.path.join(ML_DIR, 'label_encoder_res.pkl'))
le_ind  = joblib.load(os.path.join(ML_DIR, 'label_encoder_ind.pkl'))
le_comm = joblib.load(os.path.join(ML_DIR, 'label_encoder_comm.pkl'))

# Per-level feature lists — each model gets exactly the columns it was trained on
level_feature_cols = {
    level: joblib.load(os.path.join(MODEL_DIR, f'{level}_feature_cols.pkl'))
    for level in LEVELS
}

level_classes = {}
for level in LEVELS:
    level_classes[level] = joblib.load(os.path.join(ML_DIR, f'{level}_classes.pkl'))
    print(f"  {level:<20}: {len(level_feature_cols[level])} features")

print("\nEncoder class mappings:")
print(f"  L1 global     : {list(le_l1.classes_)}")
print(f"  Res subtypes  : {list(le_res.classes_)}")
print(f"  Ind subtypes  : {list(le_ind.classes_)}")
print(f"  Comm subtypes : {list(le_comm.classes_)}")

print("\nL0 class encoding (from level_classes):")
for i, cls in enumerate(level_classes['l0_res_binary']):
    print(f"  {i} → {cls}")

  l0_res_binary       : 63 features
  l1a_res_subtypes    : 62 features
  l1b_ind_binary      : 68 features
  l2a_ind_subtypes    : 69 features
  l2b_comm_binary     : 70 features
  l3a_comm_subtypes   : 89 features

Encoder class mappings:
  L1 global     : ['agricultural', 'civic', 'commercial', 'industrial', 'military', 'residential', 'transportation']
  Res subtypes  : ['MFH', 'SFH', 'TH']
  Ind subtypes  : ['manufacturing', 'storage', 'utilities']
  Comm subtypes : ['accommodation', 'food_drink', 'office', 'retail']

L0 class encoding (from level_classes):
  0 → residential
  1 → non_residential


In [17]:
print("Loading building populations...")

# 1. Fully labelled tagged buildings (stage1 L1 + L2 both known)
#    These will pass through unchanged — no prediction needed at all. 
df_train_pool = joblib.load(os.path.join(ML_DIR, 'df_train_pool.pkl'))
print(f"  df_train_pool        : {len(df_train_pool):>10,} rows")

# 2. Rule-classified buildings (stage2 L1 known, L2 may be missing)
#    These need subtype prediction only where stage2_l2 is null.
df_stage2 = joblib.load(os.path.join(ML_DIR, 'stage2_full.pkl'))
print(f"  df_stage2            : {len(df_stage2):>10,} rows")

# 3. Truly unlabelled buildings (neither stage1 nor stage2 label)
#    These need the full cascade: L0 → L1a/L1b → L2a/L2b → L3a
X_predict = joblib.load(os.path.join(ML_DIR, 'predict_raw.pkl'))
print(f"  X_predict (unlabelled): {len(X_predict):>10,} rows")

# 4. Excluded buildings (filter = garages/sheds, semi_commercial, commercial (other_service))
#    No prediction — carry through with existing labels.
df_excluded = joblib.load(os.path.join(ML_DIR, 'excluded_full.pkl'))
print(f"  df_excluded          : {len(df_excluded):>10,} rows")

# 5. Pre-sliced subtype prediction sets
#    Trainpool buildings where L1 is known but L2 is missing
X_trainpool_res  = joblib.load(os.path.join(ML_DIR, 'trainpool_l1a_res_predict.pkl'))
X_trainpool_ind  = joblib.load(os.path.join(ML_DIR, 'trainpool_l2a_ind_predict.pkl'))
X_trainpool_comm = joblib.load(os.path.join(ML_DIR, 'trainpool_l3a_comm_predict.pkl'))
print(f"  trainpool res (no L2): {len(X_trainpool_res):>10,} rows")
print(f"  trainpool ind (no L2): {len(X_trainpool_ind):>10,} rows")
print(f"  trainpool comm(no L2): {len(X_trainpool_comm):>10,} rows")

#    Stage2 buildings where L1 is known but L2 is missing
X_stage2_res  = joblib.load(os.path.join(ML_DIR, 'stage2_l1a_res_predict.pkl'))
X_stage2_ind  = joblib.load(os.path.join(ML_DIR, 'stage2_l2a_ind_predict.pkl'))
X_stage2_comm = joblib.load(os.path.join(ML_DIR, 'stage2_l3a_comm_predict.pkl'))
print(f"  stage2 res (no L2)   : {len(X_stage2_res):>10,} rows")
print(f"  stage2 ind (no L2)   : {len(X_stage2_ind):>10,} rows")
print(f"  stage2 comm(no L2)   : {len(X_stage2_comm):>10,} rows")

# Sanity check: confirm no index overlap between populations
# (every building should appear in exactly one population)
all_ids = pd.concat([
    pd.Series(df_train_pool.index, name='id'),
    pd.Series(df_stage2.index, name='id'),
    pd.Series(X_predict.index, name='id'),
    pd.Series(df_excluded.index, name='id'),
])
n_total    = len(all_ids)
n_unique   = all_ids.nunique()
n_overlap  = n_total - n_unique
print(f"\nTotal buildings across all populations: {n_total:,}")
print(f"Unique IDs                             : {n_unique:,}")
if n_overlap > 0:
    print(f"⚠️  WARNING: {n_overlap:,} duplicate IDs found across populations! Investigate before continuing.")
else:
    print("✅ No duplicate IDs — every building appears in exactly one population.")

Loading building populations...


  df_train_pool        :  9,143,593 rows
  df_stage2            :  7,270,447 rows
  X_predict (unlabelled): 18,259,833 rows
  df_excluded          :  4,128,499 rows
  trainpool res (no L2):  4,319,105 rows
  trainpool ind (no L2):    164,562 rows
  trainpool comm(no L2):    157,370 rows
  stage2 res (no L2)   :  1,433,862 rows
  stage2 ind (no L2)   :     44,813 rows
  stage2 comm(no L2)   :     24,931 rows

Total buildings across all populations: 38,802,372
Unique IDs                             : 38,802,372
✅ No duplicate IDs — every building appears in exactly one population.


In [18]:
def predict_with_proba(model, X, is_binary=True, threshold=0.5):
    """
    Run model.predict on X and return:
      - y_pred  : integer class predictions (numpy array)
      - y_proba : confidence probability for the predicted class (numpy array, float32)

    Parameters
    ----------
    model     : trained sklearn-compatible classifier
    X         : feature array or DataFrame, shape (n_samples, n_features)
    is_binary : True for binary classifiers (L0, L1b, L2b),
                False for multiclass subtype classifiers (L1a, L2a, L3a)
    """
    # Always pass numpy array to .predict() — some models behave differently
    # with DataFrames depending on version (column names may trigger warnings)
    X_arr = X.values if hasattr(X, 'values') else X


    if hasattr(model, 'predict_proba'):
        proba_matrix = model.predict_proba(X_arr)  # shape: (n_samples, n_classes)
        if is_binary:
            # For binary: probability of class 1 (non-residential / industrial / commercial)
            # We store the probability of the PREDICTED class, so:
            # - if pred=0, confidence = 1 - proba[:, 1]
            # - if pred=1, confidence = proba[:, 1]
            y_pred = (proba_matrix[:, 1] >= threshold).astype(int)
            y_proba = np.where(y_pred == 1, proba_matrix[:, 1], proba_matrix[:, 0]).astype(np.float32)
        else:
            # For multiclass: probability of the predicted class
            y_pred = model.predict(X_arr)
            y_proba = proba_matrix[np.arange(len(y_pred)), y_pred].astype(np.float32)
    else:
        # Fallback if model doesn't support predict_proba (e.g. SVM without probability=True)
        y_proba = np.full(len(y_pred), np.nan, dtype=np.float32)

    return y_pred, y_proba


print("predict_with_proba() helper defined.")

predict_with_proba() helper defined.


### Run Full Cascade on X_predict (Unlabelled Buildings)

In [ ]:

L0_THRESHOLD  = 0.350   # Residential vs Non-Residential
L1B_THRESHOLD = 0.50  # Non-Industrial vs Industrial (Set to 0.530 if you decided to use it!)
L2B_THRESHOLD = 0.400   # Other vs Commercial

t_start = time.time()

n = len(X_predict)
print(f"Running cascade on {n:,} unlabelled buildings...\n")

# Initialise a results DataFrame with the same index as X_predict.
# Every row will have final_l1 and final_l2 filled by the end of this cell.
results_predict = pd.DataFrame({
    'final_l1'     : pd.Series(dtype='object'),
    'final_l2'     : pd.Series(dtype='object'),
    'l1_confidence': pd.Series(dtype='float32'),
    'l2_confidence': pd.Series(dtype='float32'),
    'label_source' : 'ml_predicted',
}, index=X_predict.index)

# ── L0: residential vs non-residential ───────────────────────────────────────
# ALL unlabelled buildings pass through L0.
# Encoding from 06b: 0=residential, 1=non_residential  (verify against level_classes['l0_res_binary'])

print("L0: residential vs non-residential")
pred_l0, prob_l0 = predict_with_proba(
    models['l0_res_binary'], X_predict[level_feature_cols['l0_res_binary']], is_binary=True, threshold=L0_THRESHOLD)

res_mask    = pred_l0 == 0  # boolean array, length n
nonres_mask = pred_l0 == 1

print(f"  → residential    : {res_mask.sum():>9,}  ({res_mask.mean()*100:.1f}%)")
print(f"  → non-residential: {nonres_mask.sum():>9,}  ({nonres_mask.mean()*100:.1f}%)")

# Store L0 results and confidence into the results frame
results_predict.loc[X_predict.index[res_mask],    'final_l1']      = 'residential'
results_predict.loc[X_predict.index[nonres_mask], 'final_l1']      = 'non_residential_pending'
results_predict.loc[X_predict.index,              'l1_confidence'] = prob_l0

# ── L1a: residential subtypes ─────────────────────────────────────────────────
# Only buildings predicted as residential by L0 enter this model.
# Subtypes: SFH / MFH / TH 
print("\nL1a: residential subtypes")
res_ids = X_predict.index[res_mask]
X_res = X_predict.loc[res_ids, level_feature_cols['l1a_res_subtypes']]
pred_l1a, prob_l1a = predict_with_proba(models['l1a_res_subtypes'], X_res, is_binary=False)
pred_l1a_labels = le_res.inverse_transform(pred_l1a)  # integers → class name strings

results_predict.loc[X_res.index, 'final_l2']      = pred_l1a_labels
results_predict.loc[X_res.index, 'l2_confidence'] = prob_l1a
print(f"  Subtype distribution:")
for cls, cnt in zip(*np.unique(pred_l1a_labels, return_counts=True)):
    print(f"    {cls:<25}: {cnt:>9,}")

# ── L1b: industrial vs non-industrial (within non-residential) ─────────────────
# Only non-residential buildings enter this model.
# Encoding: 0=non_industrial, 1=industrial
print(f"\nL1b: industrial vs non-industrial (Threshold: {L1B_THRESHOLD})")
nonres_ids = X_predict.index[nonres_mask]
X_nonres = X_predict.loc[nonres_ids, level_feature_cols['l1b_ind_binary']]
pred_l1b, prob_l1b = predict_with_proba(models['l1b_ind_binary'], X_nonres, is_binary=True, threshold=L1B_THRESHOLD)
# Build masks AT THE NONRES LEVEL first, then translate back to global index positions.
# This is important: ind_in_nonres is a boolean array of length nonres_mask.sum(),
# NOT of length n. We will use X_nonres.index to map back.
ind_ids    = nonres_ids[pred_l1b == 1]
nonind_ids = nonres_ids[pred_l1b == 0]
ind_in_nonres    = pred_l1b == 1
nonind_in_nonres = pred_l1b == 0
print(f"  → industrial    : {ind_in_nonres.sum():>9,}")
print(f"  → non-industrial: {nonind_in_nonres.sum():>9,}")

results_predict.loc[ind_ids,   'final_l1']      = 'industrial'
results_predict.loc[nonind_ids,'final_l1']      = 'non_ind_pending'
# Overwrite l1_confidence for non-residential buildings with L1b confidence
results_predict.loc[nonres_ids,'l1_confidence'] = prob_l1b

# ── L2a: industrial subtypes ───────────────────────────────────────────────────
# Only buildings predicted as industrial by L1b enter this model.
print("\nL2a: industrial subtypes")
X_ind = X_predict.loc[ind_ids, level_feature_cols['l2a_ind_subtypes']]
pred_l2a, prob_l2a = predict_with_proba(models['l2a_ind_subtypes'], X_ind, is_binary=False)
pred_l2a_labels = le_ind.inverse_transform(pred_l2a)

results_predict.loc[ind_ids, 'final_l2']      = pred_l2a_labels
results_predict.loc[ind_ids, 'l2_confidence'] = prob_l2a
print(f"  Subtype distribution:")
for cls, cnt in zip(*np.unique(pred_l2a_labels, return_counts=True)):
    print(f"    {cls:<25}: {cnt:>9,}")

# ── L2b: commercial vs other (within non-residential, non-industrial) ─────────
# Encoding: 0=other, 1=commercial
X_nonind = X_predict.loc[nonind_ids, level_feature_cols['l2b_comm_binary']]
print(f"\nL2b: commercial vs other (Threshold: {L2B_THRESHOLD})")
pred_l2b, prob_l2b = predict_with_proba(models['l2b_comm_binary'], X_nonind, is_binary=True, threshold=L2B_THRESHOLD)
comm_ids  = nonind_ids[pred_l2b == 1]
other_ids = nonind_ids[pred_l2b == 0]


comm_in_nonind  = pred_l2b == 1
other_in_nonind = pred_l2b == 0

print(f"  → commercial: {comm_in_nonind.sum():>9,}")
print(f"  → other: {other_in_nonind.sum():>9,}")

results_predict.loc[comm_ids,  'final_l1'] = 'commercial'
results_predict.loc[other_ids, 'final_l1'] = 'other'
results_predict.loc[nonind_ids,'l1_confidence'] = prob_l2b
results_predict.loc[other_ids, 'final_l2'] = None

# ── L3a: commercial subtypes ───────────────────────────────────────────────────
# Only buildings predicted as commercial by L2b enter this model.
# Subtypes: office / retail / food_drink / accommodation
X_comm = X_predict.loc[comm_ids, level_feature_cols['l3a_comm_subtypes']]
print("\nL3a: commercial subtypes")
pred_l3a, prob_l3a = predict_with_proba(models['l3a_comm_subtypes'], X_comm, is_binary=False)
pred_l3a_labels = le_comm.inverse_transform(pred_l3a)

results_predict.loc[comm_ids, 'final_l2']      = pred_l3a_labels
results_predict.loc[comm_ids, 'l2_confidence'] = prob_l3a
print(f"  Subtype distribution:")
for cls, cnt in zip(*np.unique(pred_l3a_labels, return_counts=True)):
    print(f"    {cls:<25}: {cnt:>9,}")

# ── Final check: every row in results_predict should have a final_l1 ──────────
still_pending = results_predict['final_l1'].isin(['non_residential_pending', 'non_ind_pending'])
if still_pending.sum() > 0:
    print(f"\n⚠️  WARNING: {still_pending.sum():,} rows still have a pending L1 label — inspect these.")
else:
    print("\n✅ All rows have a final_l1 label.")

t_elapsed = time.time() - t_start
print(f"\nCascade complete in {t_elapsed/60:.1f} minutes.")
print(f"\nL1 distribution across all predicted buildings:")
print(results_predict['final_l1'].value_counts().to_string())

Running cascade on 18,259,833 unlabelled buildings...

L0: residential vs non-residential
  → residential    : 11,776,508  (64.5%)
  → non-residential: 6,483,325  (35.5%)

L1a: residential subtypes
  Subtype distribution:
    MFH                      : 1,734,521
    SFH                      : 9,576,268
    TH                       :   465,719

L1b: industrial vs non-industrial (Threshold: 0.5)
  → industrial    :   619,242
  → non-industrial: 5,864,083

L2a: industrial subtypes
  Subtype distribution:
    manufacturing            :   109,256
    storage                  :   346,148
    utilities                :   163,838

L2b: commercial vs other (Threshold: 0.4)
  → commercial: 1,216,198
  → other: 4,647,885

L3a: commercial subtypes
  Subtype distribution:
    accommodation            :   114,825
    food_drink               :   194,125
    office                   :   164,802
    retail                   :   742,446

✅ All rows have a final_l1 label.

Cascade complete in 95.9 minut

### Predict Subtypes for Stage 2 and Trainpool Buildings

In [20]:
SUBTYPE_JOBS = [
    (X_trainpool_res,  'residential', 'l1a_res_subtypes',  le_res,  'trainpool residential subtypes'),
    (X_trainpool_ind,  'industrial',  'l2a_ind_subtypes',  le_ind,  'trainpool industrial subtypes'),
    (X_trainpool_comm, 'commercial',  'l3a_comm_subtypes', le_comm, 'trainpool commercial subtypes'),
    (X_stage2_res,     'residential', 'l1a_res_subtypes',  le_res,  'stage2 residential subtypes'),
    (X_stage2_ind,     'industrial',  'l2a_ind_subtypes',  le_ind,  'stage2 industrial subtypes'),
    (X_stage2_comm,    'commercial',  'l3a_comm_subtypes', le_comm, 'stage2 commercial subtypes'),
]

subtype_results_list = []
for X_src, l1_label, model_key, encoder, description in SUBTYPE_JOBS:
    if len(X_src) == 0:
        print(f"  Skipping '{description}' — 0 rows to predict.")
        continue

    print(f"  {description}: {len(X_src):,} buildings")
    # was: X_src[feature_cols]
    pred_int, prob = predict_with_proba(models[model_key], X_src[level_feature_cols[model_key]], is_binary=False)
    pred_labels    = encoder.inverse_transform(pred_int)

    job_result = pd.DataFrame({
        'final_l1'     : l1_label,
        'final_l2'     : pred_labels,
        'l1_confidence': np.nan,
        'l2_confidence': prob,
        'label_source' : 'ml_predicted_subtype',
    }, index=X_src.index)
    subtype_results_list.append(job_result)

    for cls, cnt in zip(*np.unique(pred_labels, return_counts=True)):
        print(f"    {cls:<25}: {cnt:,}")

results_subtype = pd.concat(subtype_results_list) if subtype_results_list else pd.DataFrame()
print(f"\nTotal subtype-only predictions: {len(results_subtype):,}")

overlap = results_subtype.index.isin(results_predict.index)
if overlap.any():
    print(f"⚠️  WARNING: {overlap.sum():,} buildings appear in both results_predict and results_subtype!")
else:
    print("✅ No overlap between ml_predicted and ml_predicted_subtype sets.")

  trainpool residential subtypes: 4,319,105 buildings
    MFH                      : 733,592
    SFH                      : 3,102,888
    TH                       : 482,625
  trainpool industrial subtypes: 164,562 buildings
    manufacturing            : 49,635
    storage                  : 100,008
    utilities                : 14,919
  trainpool commercial subtypes: 157,370 buildings
    accommodation            : 10,053
    food_drink               : 10,489
    office                   : 40,279
    retail                   : 96,549
  stage2 residential subtypes: 1,433,862 buildings
    MFH                      : 312,221
    SFH                      : 1,095,299
    TH                       : 26,342
  stage2 industrial subtypes: 44,813 buildings
    manufacturing            : 15,290
    storage                  : 21,747
    utilities                : 7,776
  stage2 commercial subtypes: 24,931 buildings
    accommodation            : 79
    food_drink               : 465
    office   

### Prepare Labels for Already-Known Populations

In [21]:
# ── df_train_pool buildings that are FULLY labelled (have both L1 and L2) ──
# These were in the training set — their labels come directly from OSM tags
train_pool_full = df_train_pool[
    df_train_pool['stage1_l1'].notna() & df_train_pool['stage1_l2'].notna()
].copy()

results_trainpool_full = pd.DataFrame({
    'final_l1'     : train_pool_full['stage1_l1'].values,
    'final_l2'     : train_pool_full['stage1_l2'].values,
    'l1_confidence': np.nan,  # tag-derived, not predicted
    'l2_confidence': np.nan,
    'label_source' : 'stage1_tagged',
}, index=train_pool_full.index)

print(f"Trainpool fully labelled          : {len(results_trainpool_full):,}")

# ── df_train_pool buildings that have L1 but NOT L2 ────────────────────────
# These are in results_subtype already (predicted in Cell 7).
# We just need to confirm their L1 labels are consistent.
# (No new DataFrame needed here — already in results_subtype)

# ── df_stage2 buildings that are FULLY labelled (L1 AND L2 both known) ─────
# Stage2 rule-based classification sometimes assigned subtypes directly
# (e.g. a building tagged amenity=restaurant → commercial/food_drink)
# Also including buildings where stage2_l1 is known that is not residential/industrial/commercial

stage2_full_labels = df_stage2[
    df_stage2['stage2_l1'].notna() & df_stage2['stage2_l2'].notna()
].copy()

results_stage2_full = pd.DataFrame({
    'final_l1'     : stage2_full_labels['stage2_l1'].values,
    'final_l2'     : stage2_full_labels['stage2_l2'].values,
    'l1_confidence': np.nan,
    'l2_confidence': np.nan,
    'label_source' : 'stage2_rule',
}, index=stage2_full_labels.index)

print(f"Stage2 fully labelled (L1+L2)     : {len(results_stage2_full):,}")


# ── df_stage2 buildings with L1 only (L2 was predicted in Cell 7) ──────────
# Already in results_subtype. But note their label_source there is
# 'ml_predicted_subtype' — the L1 came from rules, L2 from ML.
# Nothing to do here.

# ── Other buildings (L1 known, no L2 model exists) ────────────
# Catch buildings from train_pool and stage2 that aren't res/ind/comm.

target_ml_classes = ['residential', 'industrial', 'commercial']

# Trainpool other
trainpool_other = df_train_pool[
    df_train_pool['stage1_l1'].notna() & 
    (~df_train_pool['stage1_l1'].isin(target_ml_classes)) & 
    df_train_pool['stage1_l2'].isna()
].copy()

results_trainpool_other = pd.DataFrame({
    'final_l1'     : trainpool_other['stage1_l1'].values,
    'final_l2'     : None, # No subtype
    'l1_confidence': np.nan,
    'l2_confidence': np.nan,
    'label_source' : 'stage1_tagged',
}, index=trainpool_other.index)

# Stage2 other
stage2_other = df_stage2[
    df_stage2['stage2_l1'].notna() & 
    (~df_stage2['stage2_l1'].isin(target_ml_classes)) & 
    df_stage2['stage2_l2'].isna()
].copy()

results_stage2_other = pd.DataFrame({
    'final_l1'     : stage2_other['stage2_l1'].values,
    'final_l2'     : None, # No subtype
    'l1_confidence': np.nan,
    'l2_confidence': np.nan,
    'label_source' : 'stage2_rule',
}, index=stage2_other.index)

print(f"Trainpool other (no subtype): {len(results_trainpool_other):,}")
print(f"Stage2 other (no subtype)   : {len(results_stage2_other):,}")

# ── 5. df_excluded (filter + semi_commercial + commercial(other_service)) ─────────────────────────────────
# These buildings were excluded from ML training and prediction.
# They keep whatever label they were assigned in the pipeline.
# stage1_l1 is the authoritative label for excluded buildings.
results_excluded = pd.DataFrame({
    'final_l1'     : df_excluded['stage1_l1'].values,
    'final_l2'     : df_excluded['stage1_l2'].values,   # may be null for semi_commercial
    'l1_confidence': np.nan,
    'l2_confidence': np.nan,
    'label_source' : 'excluded',
}, index=df_excluded.index)

print(f"Excluded (filter+semi_commercial) : {len(results_excluded):,}")
print(f"  Breakdown: {df_excluded['stage1_l1'].value_counts().to_dict()}")

Trainpool fully labelled          : 4,480,168
Stage2 fully labelled (L1+L2)     : 64,041
Trainpool other (no subtype): 22,388
Stage2 other (no subtype)   : 5,702,800
Excluded (filter+semi_commercial) : 4,128,499
  Breakdown: {'filter': 4121244, 'semi_commercial': 3692, 'commercial': 3563}


### Final Dataset

In [22]:
print("Assembling final classification table...")

# Stack all result DataFrames.
# Order doesn't matter for correctness (index is the key), but
# putting the largest populations first makes memory allocation slightly more efficient.
all_results = pd.concat([
    results_predict,          #  — full cascade ML predictions
    results_subtype,          #  — subtype-only ML predictions (stage2 + trainpool L1-only)
    results_trainpool_full,   #  — fully OSM-tagged buildings (train set)
    results_stage2_full,      #  — fully rule-classified buildings
    results_trainpool_other,  #  — train set buildings with L1 but no L2 (non-residential non-industrial non-commercial)
    results_stage2_other,     #  — stage2 buildings with L1 but no L
    results_excluded,         #  — filter + semi_commercial
], axis=0)

print(f"Total rows in final table: {len(all_results):,}")

# ── Integrity checks ──────────────────────────────────────────────────────────

gdf_bldg = gpd.read_parquet(BUILDING_FILE)

# 1. No duplicate OSM IDs
n_dupes = all_results.index.duplicated().sum()
if n_dupes > 0:
    print(f"⚠️  {n_dupes:,} duplicate IDs in final table — investigate!")
    duped_ids = all_results.index[all_results.index.duplicated(keep=False)]
    print(all_results.loc[duped_ids].head(10))
else:
    print("✅ No duplicate IDs.")

# 2. Every row has a final_l1
null_l1 = all_results['final_l1'].isna().sum()
if null_l1 > 0:
    print(f"⚠️  {null_l1:,} rows have no final_l1 — these buildings fell through the cascade without a label.")
else:
    print("✅ Every building has a final_l1 label.")

# 3. Number of rows equal to number of buildings in the original file
n_buildings = len(gdf_bldg)
if len(all_results) != n_buildings:
    print(f"⚠️  Row count mismatch: final table has {len(all_results):,} rows but original building file has {n_buildings:,} rows.")
    # print number of missing and extra IDs
    missing_ids = gdf_bldg.index.difference(all_results.index)
    extra_ids   = all_results.index.difference(gdf_bldg.index)
    print(f"  Missing IDs: {len(missing_ids):,}")
    print(f"  Extra IDs  : {len(extra_ids):,}")
else:
    print("✅ Row count matches original building file.") 

# 4. label_source audit
print("\nlabel_source breakdown:")
print(all_results['label_source'].value_counts().to_string())

Assembling final classification table...
Total rows in final table: 38,802,372
✅ No duplicate IDs.
✅ Every building has a final_l1 label.
✅ Row count matches original building file.

label_source breakdown:
label_source
ml_predicted            18259833
ml_predicted_subtype     6144643
stage2_rule              5766841
stage1_tagged            4502556
excluded                 4128499


In [23]:
print("=" * 60)
print("NATIONAL BUILDING COUNT SUMMARY — GERMANY")
print("=" * 60)

# ── L1 counts ─────────────────────────────────────────────────────────────────
l1_counts = all_results['final_l1'].value_counts()
l1_total  = l1_counts.sum()
print(f"\n{'L1 Type':<25} {'Count':>12} {'%':>8}")
print("-" * 47)
for typ, cnt in l1_counts.items():
    print(f"{str(typ):<25} {cnt:>12,} {cnt/l1_total*100:>7.2f}%")
print(f"{'TOTAL':<25} {l1_total:>12,}")

# ── L1 × L2 cross-table ───────────────────────────────────────────────────────
print(f"\n{'L1':<20} {'L2':<30} {'Count':>12} {'%':>8}")
print("-" * 72)
l1l2_counts = all_results.groupby(['final_l1', 'final_l2'], dropna=False).size()
for (l1, l2), cnt in l1l2_counts.sort_values(ascending=False).items():
    l2_str = str(l2) if pd.notna(l2) else '(no subtype)'
    print(f"{str(l1):<20} {l2_str:<30} {cnt:>12,} {cnt/l1_total*100:>7.2f}%")

# ── Confidence distribution summary ────────────────────────────────────────────
print("\nL1 confidence distribution (ML-predicted only):")
ml_only = all_results[all_results['label_source'] == 'ml_predicted']
if len(ml_only) > 0:
    print(ml_only['l1_confidence'].describe().round(3).to_string())
    low_conf = (ml_only['l1_confidence'] < 0.6).sum()
    print(f"  Low confidence predictions (< 0.60): {low_conf:,} ({low_conf/len(ml_only)*100:.1f}% of ML-predicted)")

print("\nL2 confidence distribution (ML-predicted only):")
if len(ml_only) > 0:
    l2_conf = ml_only['l2_confidence'].dropna()
    if len(l2_conf) > 0:
        print(l2_conf.describe().round(3).to_string())
        low_l2 = (l2_conf < 0.6).sum()
        print(f"  Low confidence predictions (< 0.60): {low_l2:,} ({low_l2/len(l2_conf)*100:.1f}% of subtype-predicted)")

NATIONAL BUILDING COUNT SUMMARY — GERMANY

L1 Type                          Count        %
-----------------------------------------------
residential                 20,975,630   54.06%
filter                       9,822,755   25.31%
other                        4,647,885   11.98%
commercial                   1,670,802    4.31%
industrial                     965,940    2.49%
civic                          370,217    0.95%
agricultural                   313,609    0.81%
transportation                  22,301    0.06%
military                         9,520    0.02%
semi_commercial                  3,713    0.01%
TOTAL                       38,802,372

L1                   L2                                    Count        %
------------------------------------------------------------------------


residential          SFH                              15,423,371   39.75%
filter               (no subtype)                      9,822,725   25.31%
other                (no subtype)                      4,647,885   11.98%
residential          MFH                               4,250,898   10.96%
residential          TH                                1,301,361    3.35%
commercial           retail                            1,006,280    2.59%
industrial           storage                             495,369    1.28%
industrial           utilities                           284,013    0.73%
commercial           office                              267,573    0.69%
agricultural         farm_auxiliary                      241,815    0.62%
commercial           food_drink                          237,777    0.61%
industrial           manufacturing                       186,558    0.48%
commercial           accommodation                       155,609    0.40%
civic                school           

### Save Final Output

In [25]:
# Cast dtypes before saving to minimise file size
all_results['final_l1']      = all_results['final_l1'].astype('category')
all_results['final_l2']      = all_results['final_l2'].astype('category')
all_results['label_source']  = all_results['label_source'].astype('category')
all_results['l1_confidence'] = all_results['l1_confidence'].astype(np.float32)
all_results['l2_confidence'] = all_results['l2_confidence'].astype(np.float32)

# Write to parquet
all_results.to_parquet(FINAL_OUTPUT_FILE, index=True, compression='snappy')

file_size_mb = os.path.getsize(FINAL_OUTPUT_FILE) / 1e6
print(f"Saved: {FINAL_OUTPUT_FILE}")
print(f"  Rows       : {len(all_results):,}")
print(f"  Columns    : {list(all_results.columns)}")
print(f"  File size  : {file_size_mb:.1f} MB")

# Quick reload check — confirm the file is readable and counts match
df_check = pd.read_parquet(FINAL_OUTPUT_FILE)
assert len(df_check) == len(all_results), "Row count mismatch after save/reload!"
print(f"\n✅ Reload check passed — {len(df_check):,} rows confirmed.")

Saved: /fast/home/o-olajuyigbe/osm_project/data/final/germany_buildings_classified_final.parquet
  Rows       : 38,802,372
  Columns    : ['final_l1', 'final_l2', 'l1_confidence', 'l2_confidence', 'label_source']
  File size  : 379.2 MB

✅ Reload check passed — 38,802,372 rows confirmed.
